In [ ]:

from IPython.display import clear_output

%pip install catboost -q # Cat Boost must installed before import

clear_output()

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor

In [ ]:
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
data_path = path + '/Q1_data.csv'
df = pd.read_csv(data_path)

In [ ]:
# Task 2: Write your code here:
df.head(5)

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 6))
sns.histplot(df['Delivery_Time'], kde=True, bins=30)
plt.title('Distribution of Delivery Time')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Drop the 'Order_ID' column from the data:
df = df.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Handle missing values appropriately:
missing_values = df.isnull().sum()
missing_values # The TARGET HAS NA so I will fill every column expect target i will remove the null rows in it

In [ ]:
df = df.dropna(subset=['Delivery_Time'])
missing_values = df.isnull().sum()
missing_values # I removed them now I can fill the others

In [ ]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns

df[num_cols] = SimpleImputer(strategy='median').fit_transform(df[num_cols]) # Replace numbers by Median
df[cat_cols] = SimpleImputer(strategy='most_frequent').fit_transform(df[cat_cols]) # Replace strings by Most Frequent


In [ ]:
# Task 3: Check and remove duplicates if any exist:
df = df.drop_duplicates()

In [ ]:
# Task 4:Encode categorical variables if needed:
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

In [ ]:
# Task 5: Apply feature scaling for all features (Use StandardScaler):
X = df.drop(columns=['Delivery_Time'])
y = df['Delivery_Time']

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

In [ ]:
# Task 6: Check for target imbalance and state if it is imbalanced or not:
# No need Because the task is continous (not classes to see balance or not)

In [ ]:
# Task 1: Split the dataset into features (X) and target (y):
# I did it before Scaling
X.head(2)

In [ ]:
y.head(2)

In [ ]:
# Task 2,3,4,5: Write your code here:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = [] # Empty list so I can append the losses then take the average

for train_idx, val_idx in kf.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # The model with the hyperparamters
    random_forest = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
    random_forest.fit(X_train, y_train)

    y_pred = random_forest.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred) # Calculate the MAE
    mae_scores.append(mae) # append the mae to the list

print(f"Average MAE: {np.mean(mae_scores):.2f}") # Take the average


In [ ]:
# Task 1: Plot feature importance from your trained model:
import pandas as pd
import matplotlib.pyplot as plt

feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": random_forest.feature_importances_
}).sort_values(by="importance", ascending=True)

plt.figure(figsize=(8, 6))
plt.barh(feature_importance["feature"], feature_importance["importance"])
plt.xlabel("Feature Importance")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()


In [ ]:
# Task 2: Plot predicted delivery time histogram:
plt.figure(figsize=(10, 6))
plt.hist(df['Delivery_Time'], bins=60, edgecolor='black')
plt.title('Distribution of Delivery Time')
plt.xlabel('Delivery_Time')
plt.ylabel('Count')
plt.show()

In [ ]:
# Task Bonus: Write your code here:
# Same
ensemble_mae_scores = []

for train_idx, val_idx in kf.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # I have the Random Forest Model above named "random_forest" so I dont have to do it again


    # CatBoost
    cat_boost = CatBoostRegressor(
        iterations=300,
        learning_rate=0.1,
        depth=6,
        random_seed=42,
        verbose=0
    )

    random_forest.fit(X_train, y_train)
    cat_boost.fit(X_train, y_train)

    random_forest_preds = random_forest.predict(X_val)
    cat_boost_preds = cat_boost.predict(X_val)

    # Average preds
    ensemble_preds = (random_forest_preds + cat_boost_preds) / 2

    mae = mean_absolute_error(y_val, ensemble_preds)
    ensemble_mae_scores.append(mae)

print(f"Average Ensemble MAE: {np.mean(ensemble_mae_scores):.2f}")
